# 03 - Inference Demo

A quick sanity check that the exported model (from notebook 02) actually
works end to end on a single audio file - the same logic the Flask app
in `../app/app.py` uses under the hood.


In [1]:
import pickle
import numpy as np
import librosa
from tensorflow import keras

MODEL_DIR = "../app/model"

model = keras.models.load_model(f"{MODEL_DIR}/emotion_model.h5")

with open(f"{MODEL_DIR}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

with open(f"{MODEL_DIR}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

print("Classes:", list(label_encoder.classes_))


Classes: [np.str_('angry'), np.str_('calm'), np.str_('disgust'), np.str_('fearful'), np.str_('happy'), np.str_('neutral'), np.str_('sad'), np.str_('surprised')]


In [2]:
def extract_feature(file_path, mfcc=True, chroma=True, mel=True, sr_target=22050):
    X, sr = librosa.load(file_path, sr=sr_target)
    result = np.array([])
    stft = np.abs(librosa.stft(X))
    if mfcc:
        mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sr, n_mfcc=40).T, axis=0)
        result = np.hstack((result, mfccs))
    if chroma:
        chroma_f = np.mean(librosa.feature.chroma_stft(S=stft, sr=sr).T, axis=0)
        result = np.hstack((result, chroma_f))
    if mel:
        mel_f = np.mean(librosa.feature.melspectrogram(y=X, sr=sr).T, axis=0)
        result = np.hstack((result, mel_f))
    return result


def predict_emotion(file_path):
    features = extract_feature(file_path)
    features_scaled = (features - scaler["mean"]) / scaler["std"]
    features_scaled = features_scaled.reshape(1, -1)

    probs = model.predict(features_scaled, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    for label, p in sorted(zip(label_encoder.classes_, probs), key=lambda x: -x[1]):
        print(f"  {label:>10}: {p * 100:5.2f}%")

    return pred_label


## Try it on one of the bundled sample clips

The app ships a few synthetically-generated sample clips under
`../app/static/sample_audio/` purely so the demo works out of the box
without needing the full dataset downloaded. Swap in a real RAVDESS
clip path to see it on real speech.


In [3]:
sample_path = "../app/static/sample_audio/sample_happy.wav"

prediction = predict_emotion(sample_path)
print("\nPredicted emotion:", prediction)


   surprised: 97.87%
       happy:  2.13%
       angry:  0.00%
     neutral:  0.00%
        calm:  0.00%
     disgust:  0.00%
     fearful:  0.00%
         sad:  0.00%

Predicted emotion: surprised


That's it - this is exactly what happens behind the scenes when a file is
uploaded through the web app.
